# Forecasting Model Comparison — Rolling Origin

This notebook extends the forecasting workflow with **rolling-origin evaluation** and a tree-based benchmark. The objective is to test whether predictive performance is stable across different historical forecast origins rather than depending on a single holdout period.

> The evaluation remains predictive, not causal. Every validation fold respects chronological order.

## 1. Why rolling-origin validation?

A single train/test split can make a model look unusually good or bad because the final period may contain an atypical market regime. Rolling-origin evaluation repeatedly trains on the past and evaluates on the immediately following year.

For each fold: **train on years ≤ t → predict year t+1**. No future observations are used to construct the training sample.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
QUOTATIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
TRANSACTIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
POPULATION_DIR = PROJECT_ROOT / 'data' / 'raw' / 'population'
for path in [QUOTATIONS_DIR, TRANSACTIONS_DIR, POPULATION_DIR]:
    assert path.exists(), f'Missing source directory: {path}'
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. Rebuild the annual municipality panel

The panel is reconstructed from raw repository data so the notebook is independently executable.

In [ ]:
quotation_parts = []
for path in sorted(QUOTATIONS_DIR.glob('omi_quotations_*.csv')):
    match = re.search(r'_(\d{4})_(S[12])$', path.stem)
    if not match: continue
    year, semester = int(match.group(1)), match.group(2)
    df = pd.read_csv(path, sep=';', low_memory=False)
    df.columns = [str(c).replace('\ufeff','').strip() for c in df.columns]
    required = ['Comune_ISTAT','Descr_Tipologia','Compr_min','Compr_max','Regione']
    missing = [c for c in required if c not in df.columns]
    if missing: raise KeyError(f'{path.name}: missing {missing}')
    df = df[required].copy()
    df['year'], df['semester'] = year, semester
    quotation_parts.append(df)
omi = pd.concat(quotation_parts, ignore_index=True)
omi['municipality_code'] = omi['Comune_ISTAT'].astype('string').str.strip()
omi['Compr_min'] = pd.to_numeric(omi['Compr_min'], errors='coerce')
omi['Compr_max'] = pd.to_numeric(omi['Compr_max'], errors='coerce')
omi['price_m2'] = omi[['Compr_min','Compr_max']].mean(axis=1)
residential = omi[omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)]
semester_panel = (residential.dropna(subset=['municipality_code','price_m2']).groupby(['year','semester','municipality_code'],as_index=False).agg(price_m2=('price_m2','median'),quotation_obs=('price_m2','size'),region=('Regione','first')))
annual_price = (semester_panel.groupby(['year','municipality_code'],as_index=False).agg(price_m2=('price_m2','mean'),quotation_obs=('quotation_obs','sum'),region=('region','first'),semesters_available=('semester','nunique')))

transaction_parts=[]
for folder in sorted(p for p in TRANSACTIONS_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year=int(folder.name); files=[p for p in folder.iterdir() if p.is_file()]
    lista_path=next(p for p in files if 'lista-com' in p.name.lower()); res_path=next(p for p in files if 'valori-res' in p.name.lower())
    lista=pd.read_csv(lista_path,sep=';',decimal=','); res=pd.read_csv(res_path,sep=';',decimal=',')
    lista.columns=[str(c).strip() for c in lista.columns]; res.columns=[str(c).strip() for c in res.columns]
    lista_code=next(c for c in lista.columns if re.search(r'codcom$',c,re.I)); res_code=next(c for c in res.columns if re.search(r'codcom$',c,re.I))
    ntn_candidates=[c for c in res.columns if re.fullmatch(r'NTN_?'+str(year),c,re.I)]
    if not ntn_candidates: ntn_candidates=[c for c in res.columns if re.match(r'NTN',c,re.I) and 'mq' not in c.lower()]
    ntn_col=ntn_candidates[0]
    geo_cols=[c for c in [lista_code,'Comune','Provincia','Regione'] if c in lista.columns]
    geo=lista[geo_cols].rename(columns={lista_code:'municipality_code'}).copy()
    vals=res[[res_code,ntn_col]].rename(columns={res_code:'municipality_code',ntn_col:'ntn'}).copy()
    vals['ntn']=pd.to_numeric(vals['ntn'],errors='coerce')
    if vals['municipality_code'].duplicated().any(): raise ValueError(f'{year}: duplicate municipality key')
    transaction_parts.append(geo.merge(vals,on='municipality_code',how='left',validate='one_to_one').assign(year=year))
transactions=pd.concat(transaction_parts,ignore_index=True)

population_parts=[]
for folder in sorted(p for p in POPULATION_DIR.iterdir() if p.is_dir() and p.name.isdigit()):
    year=int(folder.name); path=next(folder.glob('*_Comuni.csv'))
    df=pd.read_csv(path,sep=';',encoding='utf-8-sig',usecols=['Codice comune','Età','Totale'])
    df.columns=['municipality_code','age','population']; df['municipality_code']=df['municipality_code'].astype('string').str.strip()
    df['age']=pd.to_numeric(df['age'],errors='coerce'); df['population']=pd.to_numeric(df['population'],errors='coerce')
    population_parts.append(df.loc[df['age'].eq(999),['municipality_code','population']].assign(year=year))
population=pd.concat(population_parts,ignore_index=True)
market=annual_price.merge(transactions[['year','municipality_code','ntn']],on=['year','municipality_code'],how='left',validate='many_to_one')
market=market.merge(population,on=['year','municipality_code'],how='left',validate='many_to_one')
market=market.sort_values(['municipality_code','year']).reset_index(drop=True)
assert not market.duplicated(['year','municipality_code']).any()
g=market.groupby('municipality_code',group_keys=False)
market['price_growth']=g['price_m2'].pct_change(); market['ntn_growth']=g['ntn'].pct_change(); market['population_growth']=g['population'].pct_change()
market['price_growth_lag1']=g['price_growth'].shift(1); market['ntn_growth_lag1']=g['ntn_growth'].shift(1); market['population_growth_lag1']=g['population_growth'].shift(1)
market['price_level_lag1']=g['price_m2'].shift(1); market['ntn_level_lag1']=g['ntn'].shift(1); market['population_level_lag1']=g['population'].shift(1)
market['target_price_growth']=g['price_growth'].shift(-1)
features=['price_growth_lag1','ntn_growth_lag1','population_growth_lag1','price_level_lag1','ntn_level_lag1','population_level_lag1']
model_data=market.dropna(subset=['target_price_growth']).copy()
print(f'Model rows: {len(model_data):,}')

## 3. Rolling-origin model evaluation

We compare three models: a mean baseline, Ridge regression and Histogram Gradient Boosting. The tree model can capture nonlinear relationships, while Ridge remains the transparent benchmark.

In [ ]:
years=np.sort(model_data['year'].unique())
min_train_years=3
folds=[]

def evaluate(y_true,pred):
    return {'MAE':mean_absolute_error(y_true,pred),'RMSE':np.sqrt(mean_squared_error(y_true,pred)),'R2':r2_score(y_true,pred)}

for forecast_year in years:
    train=model_data[model_data['year'] < forecast_year].copy()
    test=model_data[model_data['year'] == forecast_year].copy()
    if len(train['year'].unique()) < min_train_years or test.empty: continue
    X_train,y_train=train[features],train['target_price_growth']; X_test,y_test=test[features],test['target_price_growth']
    baseline=DummyRegressor(strategy='mean').fit(X_train,y_train)
    ridge=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',Ridge(alpha=10.0))]).fit(X_train,y_train)
    hgb=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',HistGradientBoostingRegressor(max_iter=150,learning_rate=0.05,max_leaf_nodes=15,l2_regularization=1.0,random_state=42))]).fit(X_train,y_train)
    for name,pred in [('Baseline',baseline.predict(X_test)),('Ridge',ridge.predict(X_test)),('HistGradientBoosting',hgb.predict(X_test))]:
        row=evaluate(y_test,pred); row.update({'forecast_year':forecast_year,'model':name,'n_test':len(test)}); folds.append(row)

fold_results=pd.DataFrame(folds)
display(fold_results)

## 4. Aggregate model comparison

Average fold performance provides a more robust view than a single final-period score. Lower MAE/RMSE is better; higher R² is better.

In [ ]:
summary=(fold_results.groupby('model').agg(folds=('forecast_year','nunique'),mean_MAE=('MAE','mean'),median_MAE=('MAE','median'),mean_RMSE=('RMSE','mean'),mean_R2=('R2','mean')).sort_values('mean_MAE'))
display(summary)

fig,ax=plt.subplots(figsize=(10,5))
for model,group in fold_results.groupby('model'):
    ax.plot(group['forecast_year'],group['MAE'],marker='o',label=model)
ax.set_title('Rolling-Origin MAE by Forecast Year'); ax.set_xlabel('Forecast year'); ax.set_ylabel('MAE'); ax.legend(); fig.tight_layout(); plt.show()

## 5. Stability versus the baseline

A useful model should beat the naïve baseline across a meaningful share of forecast origins, not only on one year.

In [ ]:
baseline_mae=fold_results[fold_results['model'].eq('Baseline')][['forecast_year','MAE']].rename(columns={'MAE':'baseline_MAE'})
comparison=fold_results.merge(baseline_mae,on='forecast_year')
comparison['MAE_improvement_vs_baseline']=1-comparison['MAE']/comparison['baseline_MAE']
stability=(comparison[comparison['model'].ne('Baseline')].groupby('model')['MAE_improvement_vs_baseline'].agg(['mean','median',lambda s:(s>0).mean()]).rename(columns={'<lambda_0>':'share_folds_better'}))
display(stability)

## 6. Interpretation

The preferred model should be selected using out-of-sample error, stability across forecast origins and interpretability—not training fit alone. If the nonlinear model improves MAE consistently, the relationship between lagged market variables and future price growth may be nonlinear. This remains a predictive finding, not evidence of causality.

### Important limitations
- The historical sample may be short for high-dimensional forecasting.
- Municipalities exhibit spatial dependence.
- OMI quotations are indicative values rather than transaction prices.
- Housing supply, interest rates, income, employment, migration and macroeconomic conditions are omitted.
- Hyperparameter tuning should use nested or strictly time-aware validation; random cross-validation is inappropriate here.

### Next step
Build a production-style forecasting dataset and add rolling-window features, macroeconomic variables and municipality/regional segmentation, then compare against a carefully tuned gradient-boosting model.